# Preparing data

Notebook purpose:
- Impute missing data
- Create additional variables
- Define eligibility and create firm-level covariates for matching
- Match control firms with PSM
- Save dataset for analysis

#### 1. Set-up

In [58]:
import pandas as pd
import numpy as np
import networkx as nx

Load paths and data

In [59]:
# paths
from ipcei_spillovers.io import data_path, results_path, load_df, save_df

# logging - needed?
from ipcei_spillovers.config_logging import setup_logging
#log = setup_logging(filename="preparation.log")

In [60]:
# ----------------------------
# Load data
# ----------------------------

firm_master_ipcei_events = load_df(
    "interim", "combined", "firm_master_ipcei_events_jan26.csv", low_memory=False
)
firm_master_ipcei = load_df(
    "interim", "combined", "firm_master_ipcei_jan26.csv", low_memory=False
)
partnership_master = load_df(
    "interim", "combined", "partnership_master_ipcei_jan26.csv", low_memory=False
)
firm_month_outcomes = load_df(
    "interim", "combined", "firm_month_outcomes_jan26.csv", low_memory=False
)

#### 2. Prepare data

a) Impute missing data

In [61]:
# ========================================
# Sector, staff count and founding year
# ========================================

# ----------------------------------------
# Define corrections
# ----------------------------------------

# 1 missing sector
# 6 missing staffCount
# 44 missing foundingYear

manual_corrections = [
    {"Linkedin_name": "1s1-energy", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 10, "foundingYear": 2019},
    {"Linkedin_name": "advent-technologies-inc", "firm_type": "Manufacturer", "sector": "Hydrogen, Automotive, Other Transport", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 42, "foundingYear": 2006},
    {"Linkedin_name": "alstom", "firm_type": "Manufacturer, Service Provider", "sector": "Railway, Other Transport", "industry": "Motor Vehicle Manufacturing", "staffCount": 82359, "foundingYear": 1928},
    {"Linkedin_name": "avl", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Electric Vehicles, Battery, Hydrogen, Software, Other ICT", "industry": "Motor Vehicle Manufacturing", "staffCount": 8532, "foundingYear": 1948},
    {"Linkedin_name": "basf", "firm_type": "Manufacturer", "sector": "Chemical Industry, Agriculture, Gas, Oil", "industry": "Chemical Manufacturing", "staffCount": 67892, "foundingYear": 1865},
    {"Linkedin_name": "bmw-group", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Other ICT, Software", "industry": "Motor Vehicle Manufacturing", "staffCount": 63276, "foundingYear": 1916},
    {"Linkedin_name": "bosch", "firm_type": "Manufacturer, Service Provider", "sector": "Automotive, Software, Built Environment, Other Sectors", "industry": "Industrial Machinery Manufacturing", "staffCount": 162452, "foundingYear": 1886},
    {"Linkedin_name": "bp", "firm_type": "Manufacturer, Utility", "sector": "Oil, Gas", "industry": "Oil and Gas", "staffCount": 100500, "foundingYear": 1909},
    {"Linkedin_name": "btcomposites", "firm_type": "Manufacturer", "sector": "Wind, Hydrogen", "industry": "Machinery Manufacturing", "staffCount": 16, "foundingYear": 2009},
    {"Linkedin_name": "christofindustries", "firm_type": "Manufacturer, Service Provider", "sector": "Oil, Chemical Industry, Agriculture, Automotive, Other Construction, Mining & Metallurgy, Other Sectors", "industry": "Industrial Machinery Manufacturing", "staffCount": 329, "foundingYear": 1966},
    {"Linkedin_name": "creos-deutschland", "firm_type": "Utility", "sector": "Electricity-grid, Gas", "industry": "Utilities", "staffCount": 54, "foundingYear": 2009},
    {"Linkedin_name": "cummins-inc", "firm_type": "Manufacturer", "sector": "Other Manufacturing, Other Energy Storage", "industry": "Motor Vehicle Manufacturing", "staffCount": 51319, "foundingYear": 1919},
    {"Linkedin_name": "daimlertruckag", "firm_type": "Manufacturer", "sector": "Automotive, Other Transport", "industry": "Motor Vehicle Manufacturing", "staffCount": 8890, "foundingYear": 2021},
    {"Linkedin_name": "edpes", "firm_type": "Utility", "sector": "Wind, Solar, Hydro energy, Electricity-grid, Energy General", "industry": "Utilities", "staffCount": 796, "foundingYear": 1976},
    {"Linkedin_name": "eiffageenergiesystemes", "firm_type": "Service Provider", "sector": "Energy General", "industry": "Environmental Services", "staffCount": 14704, "foundingYear": 2017},
    {"Linkedin_name": "en-it-group", "firm_type": "Project Developer", "sector": "Wind, Solar", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 15, "foundingYear": 1993},
    {"Linkedin_name": "enbw", "firm_type": "Utility", "sector": "Energy General, Electric Vehicles", "industry": "Utilities", "staffCount": 5569, "foundingYear": 1997},
    {"Linkedin_name": "endurance-overseas-spa", "firm_type": "Manufacturer", "sector": "Automotive, Other Manufacturing", "industry": "Motor Vehicle Manufacturing", "staffCount": 142, "foundingYear": 2019},
    {"Linkedin_name": "engie", "firm_type": "Utility, Service Provider", "sector": "Energy General, Gas, Electricity-grid", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 44738, "foundingYear": 2008},
    {"Linkedin_name": "engie-belgium", "firm_type": "Utility", "sector": "Energy General, Gas, Other Renewables", "industry": "Utilities", "staffCount": 3291, "foundingYear": 1905},
    {"Linkedin_name": "eni", "firm_type": "Manufacturer, Utility", "sector": "Biofuels, Biogas, Solar, Gas, Oil, Other Renewables", "industry": "Oil and Gas", "staffCount": 31633, "foundingYear": 1953},
    {"Linkedin_name": "ewe-gasspeicher-gmbh", "firm_type": "Utility", "sector": "Gas", "industry": "Utilities", "staffCount": 90, "foundingYear": 2012},
    {"Linkedin_name": "fincantieri", "firm_type": "Manufacturer", "sector": "Shipping, Other Manufacturing", "industry": "Shipbuilding", "staffCount": 6586, "foundingYear": 1959},
    {"Linkedin_name": "fpt-industrial", "firm_type": "Manufacturer", "sector": "Automotive, Other Transport, Other Construction, Agriculture, Other Manufacturing, Energy General", "industry": "Industrial Machinery Manufacturing", "staffCount": 2864, "foundingYear": 2005},
    {"Linkedin_name": "gen-hy", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 26, "foundingYear": 2019},
    {"Linkedin_name": "hamburgerenergienetze", "firm_type": "Utility", "sector": "Electricity-grid", "industry": "Utilities", "staffCount": 771, "foundingYear": 1889},
    {"Linkedin_name": "hydrogene-de-france", "firm_type": "Manufacturer", "sector": "Hydrogen", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 3, "foundingYear": 2012},
    {"Linkedin_name": "iberdrola", "firm_type": "Utility", "sector": "Hydrogen, Wind, Solar, Other Energy Storage, Electricity-grid", "industry": "Renewable Energy Semiconductor Manufacturing", "staffCount": 26847, "foundingYear": 1901},
    {"Linkedin_name": "iqony", "firm_type": "Manufacturer, Service Provider, Project Developer", "sector": "Solar, Wind, Geothermal, Hydrogen, Other Energy Storage, Gas, Energy General", "industry": "IT Services and IT Consulting", "staffCount": 550, "foundingYear": 2022},
    {"Linkedin_name": "iveco-group", "firm_type": "Manufacturer", "sector": "Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 19778, "foundingYear": 2022},
    {"Linkedin_name": "john-cockerill", "firm_type": "Manufacturer, Service Provider", "sector": "Energy General, Iron and Steel, Solar, Wind, Hydrogen, Nuclear Energy, Other Sectors", "industry": "Industrial Machinery Manufacturing", "staffCount": 6823, "foundingYear": 2002},
    {"Linkedin_name": "katek-group", "firm_type": "Manufacturer", "sector": "Computer and Electronic Products", "industry": "Appliances, Electrical, and Electronics Manufacturing", "staffCount": 545, "foundingYear": 1976},
    {"Linkedin_name": "liofit-gmbh", "firm_type": "Service Provider", "sector": "Battery, Electric Vehicles", "industry": "Furniture and Home Furnishings Manufacturing", "staffCount": 4, "foundingYear": 2013},
    {"Linkedin_name": "nafta-a-s-", "firm_type": "Utility, Service Provider", "sector": "Gas, Oil, Other Energy Storage", "industry": "Truck Transportation", "staffCount": 194, "foundingYear": 1913},
    {"Linkedin_name": "opmobility", "firm_type": "Manufacturer", "sector": "Solar, Battery, Hydrogen, Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 6596, "foundingYear": 1946},
    {"Linkedin_name": "pkn-orlen-s.a.", "firm_type": "Utility", "sector": "Oil, Energy General", "industry": "Oil and Gas", "staffCount": 6580, "foundingYear": 1999},
    {"Linkedin_name": "prayon", "firm_type": "Manufacturer", "sector": "Chemical Industry, Battery", "industry": "Chemical Manufacturing", "staffCount": 1029, "foundingYear": 1882},
    {"Linkedin_name": "rina", "firm_type": "Service Provider", "sector": "Marine Energy, Energy General, Other Transport, Other Construction, Software, Other Sectors", "industry": "IT Services and IT Consulting", "staffCount": 7692, "foundingYear": 1861},
    {"Linkedin_name": "rwe-", "firm_type": "Utility", "sector": "Wind, Solar, Biomass, Hydrogen, Battery, Other Fossil Energy, Energy General, Coal, Nuclear Energy", "industry": "Utilities", "staffCount": 14860, "foundingYear": 1898},
    {"Linkedin_name": "sener", "firm_type": "Service Provider", "sector": "Other Transport, Other Renewables, Shipping", "industry": "IT Services and IT Consulting", "staffCount": 3815, "foundingYear": 1956},
    {"Linkedin_name": "shell", "firm_type": "Utility, Manufacturer", "sector": "Oil, Gas, Chemical Industry", "industry": "Oil and Gas", "staffCount": 166670, "foundingYear": 1907},
    {"Linkedin_name": "sibanye-stillwaters-keliber-lithium-project", "firm_type": "Mining", "sector": "Battery, Other Manufacturing, Raw Materials", "industry": "Mining", "staffCount": 131, "foundingYear": 2023},
    {"Linkedin_name": "stellantis", "firm_type": "Manufacturer", "sector": "Automotive", "industry": "Motor Vehicle Manufacturing", "staffCount": 112770, "foundingYear": 2021},
    {"Linkedin_name": "totalenergies", "firm_type": "Manufacturer, Utility", "sector": "Oil, Gas, Solar, Wind, Other Energy Storage, Shipping, Other Sectors", "industry": "Oil and Gas", "staffCount": 80116, "foundingYear": 1924},
    {"Linkedin_name": "ufihydrogen", "firm_type": "Manufacturer, Research Institute", "sector": "Hydrogen", "industry": "Industrial Machinery Manufacturing", "staffCount": 26, "foundingYear": 2023},
    {"Linkedin_name": "varta-micro-innovation-gmbh", "firm_type": "Manufacturer", "sector": "Battery", "industry": "Chemical Manufacturing", "staffCount": 8, "foundingYear": 2009},
    {"Linkedin_name": "verbund-hydrogen", "firm_type": "Utility, Project Developer", "sector": "Hydrogen", "industry": "Utilities", "staffCount": 22, "foundingYear": 2023},
    {"Linkedin_name": "vopak", "firm_type": "Service Provider", "sector": "Hydrogen, Other Energy Storage, Carbon Capture And Storage", "industry": "Truck Transportation", "staffCount": 4110, "foundingYear": 1616},
    {"Linkedin_name": "zts-v%C3%BDskum-a-v%C3%BDvoj-a-s", "firm_type": "Research Institute, Manufacturer", "sector": "Hydro energy, Battery", "industry": "Industrial Machinery Manufacturing", "staffCount": 7, "foundingYear": 1997},
]

for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
    correction_flag = f"{col}_manually_corrected"
    if correction_flag not in firm_master_ipcei.columns:
        firm_master_ipcei[correction_flag] = 0

# ----------------------------------------
# Apply corrections
# ----------------------------------------
print("\n" + "="*80)
print("APPLYING MANUAL CORRECTIONS FOR FIRM CHARACTERISTICS")
print("="*80)

n_found = 0
n_not_found = 0
corrections_applied = {
    'firm_type': 0,
    'sector': 0,
    'industry': 0,
    'staffCount': 0,
    'foundingYear': 0
}

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]

    # Find firm in dataframe
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() == 0:
        print(f"⚠️  Firm not found: {firm_name}")
        n_not_found += 1
        continue

    if mask.sum() > 1:
        print(f"⚠️  Multiple matches for: {firm_name} (using first)")

    n_found += 1

    # Track what changed for this firm
    changes = []

    # Apply each correction
    for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
        if col in correction and correction[col] is not None:
            new_value = correction[col]
            old_value = firm_master_ipcei.loc[mask, col].iloc[0]

            # Only update if different (or if old was missing)
            if pd.isna(old_value) or str(old_value).strip() == "" or old_value != new_value:
                firm_master_ipcei.loc[mask, col] = new_value
                firm_master_ipcei.loc[mask, f"{col}_manually_corrected"] = 1
                corrections_applied[col] += 1

                # Format for display
                if pd.isna(old_value) or str(old_value).strip() == "":
                    old_display = "[MISSING]"
                else:
                    old_display = str(old_value)

                changes.append(f"{col}: '{old_display}' → '{new_value}'")

    # Print summary for this firm
    if changes:
        print(f"\n✓ {firm_name}:")
        for change in changes:
            print(f"  {change}")

# ----------------------------------------
# Summary
# ----------------------------------------
print(f"\n{'='*80}")
print(f"CORRECTION SUMMARY")
print(f"{'='*80}")
print(f"Total corrections attempted: {len(manual_corrections)}")
print(f"Firms found and corrected: {n_found}")
print(f"Firms not found: {n_not_found}")
print(f"\nCorrections by field:")
for col, count in corrections_applied.items():
    print(f"  {col}: {count} firms")

if n_not_found > 0:
    print(f"\n⚠️  Firms not found in dataset - check Linkedin_name spelling:")
    for correction in manual_corrections:
        firm_name = correction["Linkedin_name"]
        if not (firm_master_ipcei["Linkedin_name"] == firm_name).any():
            print(f"  - {firm_name}")

# check for remaining missings
print(f"\n{'='*80}")
print(f"REMAINING MISSING DATA FOR IPCEI FIRMS")
print(f"{'='*80}")

ipcei_firms = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]

for col in ['firm_type', 'sector', 'industry', 'staffCount', 'foundingYear']:
    missing = ipcei_firms[col].isna() | (ipcei_firms[col] == "")
    n_missing = missing.sum()
    pct_missing = n_missing / len(ipcei_firms) * 100

    print(f"\n{col}:")
    print(f"  Missing: {n_missing} / {len(ipcei_firms)} ({pct_missing:.1f}%)")

    if n_missing > 0 and n_missing <= 10:
        print(f"  Firms still missing {col}:")
        for name in ipcei_firms[missing]["Linkedin_name"].tolist():
            print(f"    - {name}")

# ----------------------------------------
# Save summary
# ----------------------------------------
corrected_firms = firm_master_ipcei[
    (firm_master_ipcei["firm_type_manually_corrected"] == 1) |
    (firm_master_ipcei["sector_manually_corrected"] == 1) |
    (firm_master_ipcei["industry_manually_corrected"] == 1) |
    (firm_master_ipcei["staffCount_manually_corrected"] == 1) |
    (firm_master_ipcei["foundingYear_manually_corrected"] == 1)
]

correction_summary = corrected_firms[[
    "Linkedin_name",
    "firm_type", "firm_type_manually_corrected",
    "sector", "sector_manually_corrected",
    "industry", "industry_manually_corrected",
    "staffCount", "staffCount_manually_corrected",
    "foundingYear", "foundingYear_manually_corrected",
]].copy()


APPLYING MANUAL CORRECTIONS FOR FIRM CHARACTERISTICS

✓ 1s1-energy:
  foundingYear: '[MISSING]' → '2019'

✓ advent-technologies-inc:
  sector: '[MISSING]' → 'Hydrogen, Automotive, Other Transport'

✓ alstom:
  foundingYear: '[MISSING]' → '1928'

✓ avl:
  foundingYear: '[MISSING]' → '1948'

✓ basf:
  foundingYear: '[MISSING]' → '1865'

✓ bmw-group:
  foundingYear: '[MISSING]' → '1916'

✓ bosch:
  industry: 'Software Development' → 'Industrial Machinery Manufacturing'
  foundingYear: '[MISSING]' → '1886'

✓ bp:
  staffCount: '[MISSING]' → '100500'

✓ btcomposites:
  staffCount: '[MISSING]' → '16'

✓ christofindustries:
  foundingYear: '[MISSING]' → '1966'

✓ creos-deutschland:
  foundingYear: '[MISSING]' → '2009'

✓ cummins-inc:
  foundingYear: '[MISSING]' → '1919'

✓ daimlertruckag:
  foundingYear: '[MISSING]' → '2021'

✓ edpes:
  foundingYear: '[MISSING]' → '1976'

✓ eiffageenergiesystemes:
  foundingYear: '[MISSING]' → '2017'

✓ en-it-group:
  foundingYear: '[MISSING]' → '1993'

✓ en

In [62]:
# ========================================
# Country
# ========================================

print(f"Original firm_master shape: {firm_master_ipcei.shape}")
print(f"IPCEI firms (treated_any=1): {firm_master_ipcei['treated_any'].sum()}")

# ----------------------------------------
# Corrections based on LinkedIn and online research
# ----------------------------------------

manual_corrections = [
    {"Linkedin_name": "1s1-energy", "manual_country_additions": "PT, BR, US", "primary_country_new": "US"},
    {"Linkedin_name": "ae-elemental", "manual_country_additions": "PL, LU", "primary_country_new": "PL"},
    {"Linkedin_name": "air-products-europe", "manual_country_additions": "DZ, AR, BH, BE, BR, CA, CL, CN, CO, CZ, EC, FR, DE, HK, HU, IN, ID, IE, IL, IT, JP, KR, MY, MX, NL, NO, OM, PE, PL, PT, QA, RO, SA, SG, SK, ZA, ES, TH, TT, TR, TW, GB, US, AE, UA, UZ", "primary_country_new": "GB"},
    {"Linkedin_name": "airliquide", "manual_country_additions": "FR, JP, US, US, IN, DE, CN, AE, BR, CA", "primary_country_new": "FR"},
    {"Linkedin_name": "bp", "manual_country_additions": "AZ, BE, CY, DK, DE, EE, ES, FI, FR, DE, GE, GR, HU, IT, LV, LT, LU, NL, NO, PL, PT, RO, SK, RS, CH, GB, AT, CZ, BG, BR, CA, MX, TT, US, EG, IQ, KW, MR, OM, SA, SN, ZA, AE, AU, IN, ID, MY, NZ, PH, SG, KR, TH, VN, CN, JP", "primary_country_new": "GB"},
    {"Linkedin_name": "cummins-inc", "manual_country_additions": "BE, NL, DE, SE, NO, CZ, FR, IT, PL, ES, RO, TR, RS, GB, IE", "primary_country_new": "US"},
    {"Linkedin_name": "enerlis", "manual_country_additions": "PL, ES, LU", "primary_country_new": "ES"},
    {"Linkedin_name": "ferroglobe-plc", "manual_country_additions": "AR, CA, CN, FR, NO, ZA, ES, US, VE, GB", "primary_country_new": "GB"},
    {"Linkedin_name": "hydrogene-de-france", "manual_country_additions": "FR", "primary_country_new": "FR"},
    {"Linkedin_name": "nextchem", "manual_country_additions": "DE, IT, NL", "primary_country_new": "IT"},
    {"Linkedin_name": "opmobility", "manual_country_additions": "AR, AT, BE, BR, CA, CN, CZ, FR, DE, HU, IN, ID, JP, MY, MX, MA, PL, RO, RU, SK, ES, ZA, KR, CH, TH, TR, GB, US", "primary_country_new": "FR"},
    {"Linkedin_name": "titan-cement-group", "manual_country_additions": "GR, EG, US, BR, FR, IT, GB, TR, BG, RS, XK, MK, AL, IN", "primary_country_new": "GR"},
    {"Linkedin_name": "ufihydrogen", "manual_country_additions": "IT", "primary_country_new": "IT"},
    {"Linkedin_name": "uniper-se", "manual_country_additions": "DE, GB, SE, NL", "primary_country_new": "DE"},
       {"Linkedin_name": "btcomposites", "manual_country_additions": "GR", "primary_country_new": "GR"},
    {"Linkedin_name": "endurance-overseas-spa", "manual_country_additions": "IT, DE", "primary_country_new": "IT"},
    {"Linkedin_name": "hamburgerenergienetze", "manual_country_additions": "DE", "primary_country_new": "DE"},
    {"Linkedin_name": "liofit-gmbh", "manual_country_additions": "DE", "primary_country_new": "DE"},
]

# ----------------------------------------
# Helpers
# ----------------------------------------

def parse_country_string(s):
    """Convert country string to set of ISO2 codes"""
    if pd.isna(s) or s == "" or str(s).strip() == "":
        return set()
    # Clean and split
    countries = str(s).split(",")
    cleaned = set()
    for c in countries:
        c_clean = c.strip().upper()
        # Filter out invalid codes
        if c_clean and c_clean not in ["NAN", "OO", ""]:
            cleaned.add(c_clean)
    return cleaned

def country_set_to_string(country_set):
    """Convert set back to comma-separated string"""
    if not country_set:
        return ""
    return ", ".join(sorted(country_set))

# ----------------------------------------
# Application of manual corrections
# ----------------------------------------

print("\n" + "="*80)
print("APPLYING MANUAL COUNTRY CORRECTIONS")
print("="*80)

# Create flag column for tracking
if "country_manually_corrected" not in firm_master_ipcei.columns:
    firm_master_ipcei["country_manually_corrected"] = 0

n_corrections = 0
n_found = 0
n_not_found = 0

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]
    new_countries_str = correction["manual_country_additions"]
    new_primary = correction["primary_country_new"]

    # Find firm in dataframe
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() == 0:
        print(f"⚠️  Firm not found: {firm_name}")
        n_not_found += 1
        continue

    if mask.sum() > 1:
        print(f"⚠️  Multiple matches for: {firm_name} (using first)")

    n_found += 1

    # Get existing countries
    existing = parse_country_string(firm_master_ipcei.loc[mask, "countries"].iloc[0])

    # Parse new countries from manual research
    new_countries = parse_country_string(new_countries_str)

    # Combine (union)
    all_countries = existing | new_countries

    # Update countries string
    old_countries_str = firm_master_ipcei.loc[mask, "countries"].iloc[0]
    new_countries_str_final = country_set_to_string(all_countries)

    firm_master_ipcei.loc[mask, "countries"] = new_countries_str_final
    firm_master_ipcei.loc[mask, "n_countries"] = len(all_countries)

    # Update primary_country if provided and different
    old_primary = firm_master_ipcei.loc[mask, "primary_country"].iloc[0]

    if new_primary and str(new_primary).strip():
        new_primary_clean = str(new_primary).strip().upper()
        if new_primary_clean not in ["NAN", "OO", ""]:
            # Only update if different
            if pd.isna(old_primary) or str(old_primary).strip().upper() != new_primary_clean:
                firm_master_ipcei.loc[mask, "primary_country"] = new_primary_clean
                primary_changed = True
            else:
                primary_changed = False
        else:
            primary_changed = False
    else:
        primary_changed = False

    # Mark as manually corrected
    firm_master_ipcei.loc[mask, "country_manually_corrected"] = 1

    # Log the change
    print(f"\n✓ {firm_name}:")
    print(f"  Old countries: '{old_countries_str}' ({len(existing)} countries)")
    print(f"  New countries: '{new_countries_str_final}' ({len(all_countries)} countries)")
    if primary_changed:
        print(f"  Primary country: '{old_primary}' → '{new_primary_clean}'")

    n_corrections += 1

print(f"\n{'='*80}")
print(f"CORRECTION SUMMARY")
print(f"{'='*80}")
print(f"Total corrections attempted: {len(manual_corrections)}")
print(f"Firms found and corrected: {n_found}")
print(f"Firms not found: {n_not_found}")

if n_not_found > 0:
    print("\n⚠️  Firms not found in dataset - check Linkedin_name spelling:")
    for correction in manual_corrections:
        firm_name = correction["Linkedin_name"]
        if not (firm_master_ipcei["Linkedin_name"] == firm_name).any():
            print(f"  - {firm_name}")

# ----------------------------------------
# Summary stats
# ----------------------------------------

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nTotal firms: {len(firm_master_ipcei)}")
print(f"IPCEI firms (treated_any=1): {firm_master_ipcei['treated_any'].sum()}")

n_manual = firm_master_ipcei["country_manually_corrected"].sum()
print(f"Firms with manual country corrections: {n_manual}")

# Country completeness
missing_countries = (
    firm_master_ipcei["countries"].isna() |
    (firm_master_ipcei["countries"] == "") |
    (firm_master_ipcei["countries"].str.upper().isin(["NAN", "OO"]))
)

n_missing = missing_countries.sum()
pct_missing = n_missing / len(firm_master_ipcei) * 100
print(f"\nFirms with missing 'countries': {n_missing} ({pct_missing:.1f}%)")

# IPCEI firms with missing countries
ipcei_missing = firm_master_ipcei[
    (firm_master_ipcei["treated_any"] == 1) & missing_countries
]

if len(ipcei_missing) > 0:
    print(f"\n⚠️  IPCEI firms still missing countries: {len(ipcei_missing)}")
    print("\nFirms that still need correction:")
    for idx, row in ipcei_missing.iterrows():
        print(f"\n  {row['Linkedin_name']}:")
        print(f"    current countries: '{row['countries']}'")
        print(f"    primary_country: {row['primary_country']}")
        print(f"    hq_country: {row['hq_country']}")
else:
    print(f"\n✓ All IPCEI firms have country information")

# Distribution of n_countries for IPCEI firms
print(f"\nCountry distribution for IPCEI firms:")
ipcei_firms = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]
country_dist = ipcei_firms["n_countries"].value_counts().sort_index()
for n, count in country_dist.items():
    print(f"  {int(n)} countries: {count} firms")

# Average number of countries
mean_countries = ipcei_firms["n_countries"].mean()
print(f"\nAverage countries per IPCEI firm: {mean_countries:.1f}")

# Manually corrected IPCEI firms
ipcei_manual = ipcei_firms[ipcei_firms["country_manually_corrected"] == 1]
print(f"IPCEI firms with manual corrections: {len(ipcei_manual)} ({len(ipcei_manual)/len(ipcei_firms)*100:.1f}%)")

# ----------------------------------------
# Save summary
# ----------------------------------------

changes_summary = []

for correction in manual_corrections:
    firm_name = correction["Linkedin_name"]
    mask = firm_master_ipcei["Linkedin_name"] == firm_name

    if mask.sum() > 0:
        row = firm_master_ipcei.loc[mask].iloc[0]
        changes_summary.append({
            "Linkedin_name": firm_name,
            "countries_final": row["countries"],
            "n_countries": row["n_countries"],
            "primary_country": row["primary_country"],
            "manually_corrected": row["country_manually_corrected"]
        })

changes_df = pd.DataFrame(changes_summary)

save_df(
    changes_df,
    "interim",
    "ipcei",
    "ipcei_firms_country_summary.csv"
)

print(f"\n✓ Saved correction summary to ipcei_firms_country_summary.csv")

Original firm_master shape: (239782, 36)
IPCEI firms (treated_any=1): 150

APPLYING MANUAL COUNTRY CORRECTIONS

✓ 1s1-energy:
  Old countries: 'nan' (0 countries)
  New countries: 'BR, PT, US' (3 countries)
  Primary country: 'nan' → 'US'

✓ ae-elemental:
  Old countries: 'nan' (0 countries)
  New countries: 'LU, PL' (2 countries)
  Primary country: 'nan' → 'PL'

✓ air-products-europe:
  Old countries: 'GB' (1 countries)
  New countries: 'AE, AR, BE, BH, BR, CA, CL, CN, CO, CZ, DE, DZ, EC, ES, FR, GB, HK, HU, ID, IE, IL, IN, IT, JP, KR, MX, MY, NL, NO, OM, PE, PL, PT, QA, RO, SA, SG, SK, TH, TR, TT, TW, UA, US, UZ, ZA' (46 countries)

✓ airliquide:
  Old countries: 'BR, CA, FR, JP, NL' (5 countries)
  New countries: 'AE, BR, CA, CN, DE, FR, IN, JP, NL, US' (10 countries)

✓ bp:
  Old countries: 'BR, GB, US' (3 countries)
  New countries: 'AE, AT, AU, AZ, BE, BG, BR, CA, CH, CN, CY, CZ, DE, DK, EE, EG, ES, FI, FR, GB, GE, GR, HU, ID, IN, IQ, IT, JP, KR, KW, LT, LU, LV, MR, MX, MY, NL, N

In [63]:
# ============================================================
# Save as corrected version
# ============================================================

save_df(
    firm_master_ipcei,
    "interim",
    "combined",
    "firm_master_ipcei_jan26_corrected.csv"  # Overwrite
)

print("✓ Saved: firm_master_ipcei_jan26_corrected.csv")

✓ Saved: firm_master_ipcei_jan26_corrected.csv


In [64]:
# Load corrected firm master
firm_master_ipcei = load_df(
    "interim", "combined", "firm_master_ipcei_jan26_corrected.csv", low_memory=False
)

b) Transform variables

In [65]:
# ========================================
# Log staff count
# ========================================
firm_master_ipcei["log_staffCount"] = np.log(
    firm_master_ipcei["staffCount"].fillna(0) + 1
)

# ----------------------------------------
# Create staffCount buckets (EU SME classification)
# ----------------------------------------

def assign_size_bucket(staff_count):
    """
    Assign firm size bucket based on EU SME definition.

    Categories:
    - Micro: 1-9 employees
    - Small: 10-49 employees
    - Medium: 50-249 employees
    - Large: 250+ employees
    """
    if pd.isna(staff_count) or staff_count == 0:
        return "Unknown"
    elif staff_count < 10:
        return "Micro (1-9)"
    elif staff_count < 50:
        return "Small (10-49)"
    elif staff_count < 250:
        return "Medium (50-249)"
    else:
        return "Large (250+)"

firm_master_ipcei["staffCount_bucket"] = firm_master_ipcei["staffCount"].apply(assign_size_bucket)

# Create numeric version for continuous matching
size_bucket_order = {
    "Unknown": 0,
    "Micro (1-9)": 1,
    "Small (10-49)": 2,
    "Medium (50-249)": 3,
    "Large (250+)": 4
}
firm_master_ipcei["staffCount_bucket_num"] = firm_master_ipcei["staffCount_bucket"].map(size_bucket_order)

print("\nFirm size distribution:")
print(firm_master_ipcei["staffCount_bucket"].value_counts().sort_index())

print("\nTreated firms - size distribution:")
print(firm_master_ipcei[firm_master_ipcei["treated_any"]==1]["staffCount_bucket"].value_counts())


Firm size distribution:
staffCount_bucket
Large (250+)       38866
Medium (50-249)    40299
Micro (1-9)        38423
Small (10-49)      50848
Unknown            71346
Name: count, dtype: int64

Treated firms - size distribution:
staffCount_bucket
Large (250+)       90
Medium (50-249)    33
Small (10-49)      20
Micro (1-9)         7
Name: count, dtype: int64


In [66]:
# ========================================
# Firm age
# ========================================
REFERENCE_YEAR = 2024 # later baseline year to capture firms founded after 2020

firm_master_ipcei["age"] = REFERENCE_YEAR - firm_master_ipcei["foundingYear"]
firm_master_ipcei.loc[firm_master_ipcei["age"] < 0, "age"] = np.nan

# Drop firms with foundingYear < 400 (errors)
# Keep historical entities with foundingYear >= 400 (e.g., République Française founded 496)
firm_master_ipcei.loc[firm_master_ipcei["foundingYear"] < 500, "age"] = np.nan

print(f"Firms with cleaned age (set to NaN): {firm_master_ipcei['age'].isna().sum()}")

Firms with cleaned age (set to NaN): 71145


c) Create new variables

In [67]:
# ============================================================
# GEOGRAPHIC OPERATIONS
# ============================================================

print("\n" + "="*80)
print("COMPUTING GEOGRAPHIC VARIABLES")
print("="*80)

EU_CODES = {
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR",
    "HU","IE","IT","LV","LT","LU","MT","NL","PL","PT","RO","SK",
    "SI","ES","SE"
}

years = list(range(2020, 2026))

# ── EU operation dummy ──────────────────────────────────────────────
def has_eu_op(country_str):
    if pd.isna(country_str):
        return 0
    countries = {c.strip().upper() for c in str(country_str).split(",")}
    return int(len(countries & EU_CODES) > 0)

firm_master_ipcei["has_eu_operation"] = firm_master_ipcei["countries"].apply(has_eu_op)

# ── log n_countries ─────────────────────────────────────────────────
firm_master_ipcei["log_n_countries"] = np.log(firm_master_ipcei["n_countries"] + 1)

# ── Primary country is EU ────────────────────────────────────────────────
def primary_is_eu(primary_str):
    if pd.isna(primary_str) or str(primary_str).strip() in ("", "nan"):
        return 0
    return int(str(primary_str).strip().upper() in EU_CODES)

firm_master_ipcei["primary_eu"] = firm_master_ipcei["primary_country"].apply(primary_is_eu)

# ── EU cross-border collaboration counts (annual) ────────────────────────

p = partnership_master.copy()
p["post_date"] = pd.to_datetime(p["post_date"], errors="coerce")
p = p.dropna(subset=["post_date", "source", "target", "collaboration_id"])
p["year"]  = p["post_date"].dt.year
p["month"] = p["post_date"].dt.to_period("M").dt.to_timestamp()

# Build country maps from firm_master_ipcei
def parse_countries(x):
    if pd.isna(x) or str(x).strip() in ("", "nan"):
        return set()
    return {c.strip().upper() for c in str(x).split(",")}

def parse_primary(x):
    if pd.isna(x) or str(x).strip() in ("", "nan"):
        return None
    return str(x).strip().upper()

country_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["countries"].apply(parse_countries)
))
primary_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["primary_country"].apply(parse_primary)
))

# EU flags per endpoint (any EU operation)
p["src_eu"] = p["source"].map(country_map).apply(
    lambda x: bool(x & EU_CODES) if isinstance(x, set) else False
)
p["tgt_eu"] = p["target"].map(country_map).apply(
    lambda x: bool(x & EU_CODES) if isinstance(x, set) else False
)

# EU flags per endpoint (primary country)
p["src_eu_primary"] = p["source"].map(primary_map).apply(
    lambda x: x in EU_CODES if x else False
)
p["tgt_eu_primary"] = p["target"].map(primary_map).apply(
    lambda x: x in EU_CODES if x else False
)
p["src_primary_val"] = p["source"].map(primary_map)
p["tgt_primary_val"] = p["target"].map(primary_map)

# Classification flags
p["is_eu_ecosystem"]    = p["src_eu"] & p["tgt_eu"]
p["is_eu_primary"]      = p["src_eu_primary"] & p["tgt_eu_primary"]
p["is_eu_crossborder"]  = (
    p["src_eu_primary"] &
    p["tgt_eu_primary"] &
    (p["src_primary_val"] != p["tgt_primary_val"])
)
p["is_non_eu"] = (
    (p["src_eu_primary"] & ~p["tgt_eu_primary"]) |
    (~p["src_eu_primary"] & p["tgt_eu_primary"])
)
p["is_domestic"] = (
    p["src_eu_primary"] &
    p["tgt_eu_primary"] &
    (p["src_primary_val"] == p["tgt_primary_val"])
)

print("✓ Partnership EU flags computed")
print(f"  EU ecosystem:   {p['is_eu_ecosystem'].sum():,} partnerships")
print(f"  EU primary:     {p['is_eu_primary'].sum():,} partnerships")
print(f"  EU crossborder: {p['is_eu_crossborder'].sum():,} partnerships")
print(f"  Non-EU:         {p['is_non_eu'].sum():,} partnerships")
print(f"  Domestic:         {p['is_domestic'].sum():,} partnerships")

# Annual EU collab counts per firm
def _stack_annual(df_sub, col_name):
    src = df_sub[["year","collaboration_id","source"]].rename(columns={"source":"Linkedin_name"})
    tgt = df_sub[["year","collaboration_id","target"]].rename(columns={"target":"Linkedin_name"})
    ft  = pd.concat([src, tgt], ignore_index=True)
    ft  = ft[ft["year"].isin(years)]
    counts = (
        ft.groupby(["Linkedin_name","year"])["collaboration_id"]
          .nunique().reset_index(name=col_name)
    )
    wide = counts.pivot(index="Linkedin_name", columns="year", values=col_name).fillna(0)
    wide.columns = [f"{col_name}_{int(c)}" for c in wide.columns]
    return wide.reset_index()

eu_eco_df  = _stack_annual(p[p["is_eu_ecosystem"]],   "eu_ecosystem_collabs")
eu_prim_df = _stack_annual(p[p["is_eu_primary"]],     "eu_primary_collabs")
eu_cb_df   = _stack_annual(p[p["is_eu_crossborder"]], "eu_crossborder_collabs")
non_eu_df  = _stack_annual(p[p["is_non_eu"]],         "non_eu_collabs")
domestic_df = _stack_annual(p[p["is_domestic"]], "domestic_collabs")

for df_add in [eu_eco_df, eu_prim_df, eu_cb_df, non_eu_df, domestic_df]:
    firm_master_ipcei = firm_master_ipcei.merge(df_add, on="Linkedin_name", how="left")

firm_master_ipcei = firm_master_ipcei.fillna(0)

# Baseline focus variables (2020)
firm_master_ipcei["eu_crossborder_collabs_pre"] = (
    firm_master_ipcei.get("eu_crossborder_collabs_2020", 0)
)

print("✓ EU collaboration variables merged into firm_master_ipcei")


COMPUTING GEOGRAPHIC VARIABLES
✓ Partnership EU flags computed
  EU ecosystem:   1,623,599 partnerships
  EU primary:     1,470,846 partnerships
  EU crossborder: 615,210 partnerships
  Non-EU:         1,050,682 partnerships
  Domestic:         855,636 partnerships
✓ EU collaboration variables merged into firm_master_ipcei


In [68]:
# ==========================================
# NACE 1-digit proxy classification
# ==========================================
# Field priority: firm_type (unambiguous) > industry > firm_type (ambiguous) > sector
# firm_type is the most consistently applied field.
# industry is specific but unreliable for large conglomerates.
# sector only describes activity domains
# ==========================================

def row_get(val):
    """Return clean string or empty string if missing/zero."""
    if val is None:
        return ""
    s = str(val).strip()
    return "" if s in ("0", "nan", "none", "") else s

def classify_firm_type(t):
    """
    Classify from firm_type field — unambiguous cases only.
    Returns None if ambiguous (e.g. Manufacturer + Service Provider).
    """
    t = str(t).lower().strip()
    if not t or t in ("nan", "none", "0", ""):
        return None
    if "utility" in t:
        return "D — Energy"
    if "mining" in t:
        return "B — Mining"
    if "research institute" in t:
        return "M — Professional"
    if "manufacturer" in t and "service" not in t:
        return "C — Manufacturing"
    return None

def classify_text(t):
    """Classify from industry or sector text field."""
    t = str(t).lower().strip()
    if not t or t in ("nan", "none", "0", ""):
        return None

    if any(k in t for k in ["agriculture", "farming"]):
        return "A — Agriculture"
    if any(k in t for k in ["mining", "metals", "raw material"]):
        return "B — Mining"

    if any(k in t for k in [
        "manufacturing", "machinery", "industrial",
        "chemical", "electronics", "electrical",
        "appliances", "materials", "battery", "semiconductor",
        "shipbuilding"
    ]):
        return "C — Manufacturing"
    if any(k in t for k in [
        "utility", "utilities", "energy", "power", "oil", "gas",
        "renewable", "electricity"
    ]):
        return "D — Energy"
    if any(k in t for k in ["environmental", "water", "waste"]):
        return "E — Environmental"
    if any(k in t for k in ["construction", "infrastructure"]):
        return "F — Construction"
    if any(k in t for k in ["wholesale", "retail", "trade"]):
        return "G — Trade"
    if any(k in t for k in [
        "transport", "automotive", "motor vehicle",
        "mobility", "shipping", "aviation", "rail", "truck"
    ]):
        return "H — Transport"
    if any(k in t for k in [
        "software", "information technology", "it services",
        "it consulting", "digital"
    ]):
        return "J — Information"
    if any(k in t for k in ["financial", "bank", "investment", "financing"]):
        return "K — Finance"
    if any(k in t for k in [
        "research", "biotechnology", "nanotechnology",
        "scientific", "engineering"
    ]):
        return "M — Professional"
    if any(k in t for k in ["government", "public", "administration"]):
        return "O — Public"
    if any(k in t for k in ["consulting", "advisory"]):
        return "M — Professional"

    return None

def classify_nace1_full(industry, firm_type=None, sector=None):

    ft = str(row_get(firm_type)).lower()

    # Priority 1: unambiguous firm_type
    result = classify_firm_type(row_get(firm_type))
    if result:
        return result

    # Priority 2: ambiguous firm_type — Manufacturer wins over industry
    # A firm calling itself a Manufacturer is a manufacturer regardless
    # of what LinkedIn industry field says
    if "manufacturer" in ft:
        return "C — Manufacturing"
    if "project developer" in ft:
        return "D — Energy"

    # Priority 3: industry field (only for non-manufacturers)
    result = classify_text(row_get(industry))
    if result:
        return result

    # Priority 4: sector (last resort)
    result = classify_text(row_get(sector))
    if result:
        return result

    return "Other"


print("\nApplying FULL NACE proxy classification...")

firm_master_ipcei["nace1_proxy"] = firm_master_ipcei.apply(
    lambda row: classify_nace1_full(
        row_get(row.get("industry_manually_corrected") or row.get("industry")),
        row_get(row.get("firm_type_manually_corrected") or row.get("firm_type")),
        row_get(row.get("sector_manually_corrected") or row.get("sector"))
    ),
    axis=1
)

print("✓ NACE full classification complete")

print("\nDistribution:")
print(firm_master_ipcei["nace1_proxy"].value_counts())

print("\nTreated distribution:")
print(
    firm_master_ipcei.loc[
        firm_master_ipcei["treated_any"] == 1,
        "nace1_proxy"
    ].value_counts()
)

print("\nOverlap check:")
print(
    pd.crosstab(
        firm_master_ipcei["nace1_proxy"],
        firm_master_ipcei["treated_any"]
    )
)

# Verify previously suspect firms
suspect = ["basf", "christofindustries", "fpt-industrial",
           "bosch", "hybritdevelopment", "luxcara",
           "rostock-port", "solarfoods", "everfuel"]

print("\nSuspect firms after fix:")
print(
    firm_master_ipcei[
        firm_master_ipcei["Linkedin_name"].isin(suspect)
    ][[
        "Linkedin_name", "industry", "firm_type",
        "sector", "nace1_proxy"
    ]].to_string()
)


Applying FULL NACE proxy classification...
✓ NACE full classification complete

Distribution:
nace1_proxy
C — Manufacturing    70422
Other                53856
D — Energy           21282
M — Professional     19447
J — Information      15667
O — Public           14538
H — Transport        13431
K — Finance           9310
F — Construction      8405
E — Environmental     5100
G — Trade             4421
B — Mining            2202
A — Agriculture       1701
Name: count, dtype: int64

Treated distribution:
nace1_proxy
C — Manufacturing    92
D — Energy           42
M — Professional      8
E — Environmental     2
J — Information       2
H — Transport         2
K — Finance           1
B — Mining            1
Name: count, dtype: int64

Overlap check:
treated_any            0   1
nace1_proxy                 
A — Agriculture     1701   0
B — Mining          2201   1
C — Manufacturing  70330  92
D — Energy         21240  42
E — Environmental   5098   2
F — Construction    8405   0
G — Trade      

In [69]:
# ==========================================
# Collapse NACE for matching
# ==========================================

collapse_map = {
    "C — Manufacturing": "Production",
    "B — Mining": "Production",

    "D — Energy": "Energy system",
    "E — Environmental": "Energy system",

    "H — Transport": "Transport",

    "J — Information": "Knowledge economy",
    "K — Finance": "Knowledge economy",
    "M — Professional": "Knowledge economy",

    "A — Agriculture": "Primary sector",

    "F — Construction": "Services / Public",
    "G — Trade": "Services / Public",
    "O — Public": "Services / Public",
    "Other": "Services / Public"
}

firm_master_ipcei["nace1_psm"] = firm_master_ipcei["nace1_proxy"].map(collapse_map)

print("\nPSM sector distribution:")
print(firm_master_ipcei["nace1_psm"].value_counts())

print("\nTreated distribution:")
print(
    firm_master_ipcei.loc[firm_master_ipcei["treated_any"] == 1, "nace1_psm"]
    .value_counts()
)


PSM sector distribution:
nace1_psm
Services / Public    81220
Production           72624
Knowledge economy    44424
Energy system        26382
Transport            13431
Primary sector        1701
Name: count, dtype: int64

Treated distribution:
nace1_psm
Production           93
Energy system        44
Knowledge economy    11
Transport             2
Name: count, dtype: int64


In [70]:
# ============================================================
# NACE 2-digit proxy from LinkedIn industry  (corrected sector variable)
# ============================================================
# Replaces the old 13-entry nace2_industry map. Resolution rule:
#   - IPCEI-relevant sections (B,C,D,E,H,J,M) split to NACE-2 division;
#   - split-section orphans fall to a per-section residual token;
#   - non-split sections keep the NACE-1 letter.
# Unmapped / empty industry -> NaN (genuinely unclassified). A pure
# NACE-1-letter collapse (nace2_proxy_nace1) is derived for robustness.
# Drives ONLY the cross-sector outcome; not matching, not other outcomes.
# ============================================================

print("\n" + "="*80)
print("COMPUTING NACE2_PROXY")
print("="*80)

# industry string -> (NACE-1 section letter, NACE-2 division | None)
industry_to_sector = {
    # A — Agriculture
    "Farming": ("A", None),
    "Fisheries": ("A", None),
    "Ranching": ("A", None),
    # B — Mining
    "Mining": ("B", "07 — Mining"),
    # C — Manufacturing
    "Renewable Energy Semiconductor Manufacturing": ("C", "26 — Electronics Manufacturing"),
    "Motor Vehicle Manufacturing": ("C", "29 — Motor Vehicle Manufacturing"),
    "Industrial Machinery Manufacturing": ("C", "28 — Machinery Manufacturing"),
    "Appliances, Electrical, and Electronics Manufacturing": ("C", "26 — Electronics Manufacturing"),
    "Machinery Manufacturing": ("C", "28 — Machinery Manufacturing"),
    "Manufacturing": ("C", None),
    "Chemical Manufacturing": ("C", "20 — Chemical Manufacturing"),
    "Food and Beverage Manufacturing": ("C", None),
    "Aviation and Aerospace Component Manufacturing": ("C", None),
    "Automation Machinery Manufacturing": ("C", "28 — Machinery Manufacturing"),
    "Shipbuilding": ("C", None),
    "Defense and Space Manufacturing": ("C", None),
    "Plastics Manufacturing": ("C", None),
    "Paper and Forest Product Manufacturing": ("C", None),
    "Semiconductor Manufacturing": ("C", "26 — Electronics Manufacturing"),
    "Textile Manufacturing": ("C", None),
    "Computers and Electronics Manufacturing": ("C", "26 — Electronics Manufacturing"),
    "Packaging and Containers Manufacturing": ("C", None),
    "Medical Equipment Manufacturing": ("C", None),   # MED
    "Pharmaceutical Manufacturing": ("C", None),
    "Beverage Manufacturing": ("C", None),
    "Furniture and Home Furnishings Manufacturing": ("C", None),
    "Glass, Ceramics and Concrete Manufacturing": ("C", None),
    "Railroad Equipment Manufacturing": ("C", None),
    "Printing Services": ("C", None),
    "Personal Care Product Manufacturing": ("C", None),   # MED
    "Computer Hardware Manufacturing": ("C", "26 — Electronics Manufacturing"),
    "Sporting Goods Manufacturing": ("C", None),
    "Dairy Product Manufacturing": ("C", None),
    "Tobacco Manufacturing": ("C", None),
    # D — Energy
    "Utilities": ("D", "35 — Utilities"),
    "Oil and Gas": ("D", "06 — Oil and Gas"),
    # E — Environmental
    "Environmental Services": ("E", "38 — Environmental Services"),
    # F — Construction
    "Construction": ("F", None),
    # G — Trade
    "Retail": ("G", None),
    "International Trade and Development": ("G", None),
    "Wholesale Building Materials": ("G", None),
    "Wholesale": ("G", None),
    "Retail Apparel and Fashion": ("G", None),
    "Retail Office Equipment": ("G", None),
    "Retail Luxury Goods and Jewelry": ("G", None),
    "Wholesale Import and Export": ("G", None),
    "Retail Groceries": ("G", None),
    "Retail Art Supplies": ("G", None),
    # H — Transport
    "Truck Transportation": ("H", "49 — Land Transport"),
    "Transportation, Logistics, Supply Chain and Storage": ("H", "49 — Land Transport"),   # MED
    "Airlines and Aviation": ("H", None),
    "Maritime Transportation": ("H", None),
    "Freight and Package Transportation": ("H", "49 — Land Transport"),   # MED
    # I — Accommodation & food
    "Hospitality": ("I", None),
    "Food and Beverage Services": ("I", None),
    "Travel Arrangements": ("I", None),
    "Restaurants": ("I", None),
    # J — Information
    "IT Services and IT Consulting": ("J", "62 — IT Services"),
    "Software Development": ("J", "62 — IT Services"),
    "Media Production": ("J", None),
    "Telecommunications": ("J", None),
    "Book and Periodical Publishing": ("J", None),
    "Newspaper Publishing": ("J", None),
    "Online Audio and Video Media": ("J", None),
    "Broadcast Media Production and Distribution": ("J", None),
    "Information Services": ("J", None),
    "Technology, Information and Internet": ("J", "62 — IT Services"),   # MED
    "Computer Games": ("J", None),
    "Movies, Videos, and Sound": ("J", None),
    "Animation and Post-production": ("J", None),
    "Wireless Services": ("J", None),
    "Computer Networking Products": ("J", None),
    # K — Finance
    "Financial Services": ("K", None),
    "Venture Capital and Private Equity Principals": ("K", None),
    "Banking": ("K", None),
    "Insurance": ("K", None),
    "Investment Banking": ("K", None),
    "Investment Management": ("K", None),
    "Capital Markets": ("K", None),
    # L — Real estate
    "Real Estate": ("L", None),
    "Leasing Non-residential Real Estate": ("L", None),
    # M — Professional & scientific
    "Research Services": ("M", "72 — Research Services"),
    "Business Consulting and Services": ("M", None),
    "Architecture and Planning": ("M", None),
    "Civil Engineering": ("M", None),
    "Law Practice": ("M", None),
    "Biotechnology Research": ("M", "72 — Research Services"),
    "Advertising Services": ("M", None),
    "Design Services": ("M", None),
    "Think Tanks": ("M", None),
    "Public Relations and Communications Services": ("M", None),
    "Nanotechnology Research": ("M", "72 — Research Services"),
    "Computer and Network Security": ("M", None),   # MED
    "Accounting": ("M", None),
    "Market Research": ("M", None),
    "Photography": ("M", None),
    "Legal Services": ("M", None),
    "Writing and Editing": ("M", None),
    "Outsourcing and Offshoring Consulting": ("M", None),
    "Strategic Management Services": ("M", None),
    "Graphic Design": ("M", None),
    "Alternative Dispute Resolution": ("M", None),
    "Translation and Localization": ("M", None),
    # N — Administrative
    "Events Services": ("N", None),
    "Facilities Services": ("N", None),
    "Staffing and Recruiting": ("N", None),
    "Executive Offices": ("N", None),
    "Human Resources Services": ("N", None),
    "Security and Investigations": ("N", None),
    "Warehousing and Storage": ("N", None),   # MED
    # O — Public
    "Government Administration": ("O", None),
    "Government Relations Services": ("O", None),
    "International Affairs": ("O", None),
    "Political Organizations": ("O", None),
    "Public Safety": ("O", None),
    "Public Policy Offices": ("O", None),
    "Armed Forces": ("O", None),
    "Law Enforcement": ("O", None),
    "Legislative Offices": ("O", None),
    "Administration of Justice": ("O", None),
    # P — Education
    "Higher Education": ("P", None),
    "Education Administration Programs": ("P", None),
    "Professional Training and Coaching": ("P", None),
    "Primary and Secondary Education": ("P", None),
    "E-Learning Providers": ("P", None),
    "Libraries": ("P", None),
    # Q — Health
    "Hospitals and Health Care": ("Q", None),
    "Wellness and Fitness Services": ("Q", None),
    "Individual and Family Services": ("Q", None),
    "Mental Health Care": ("Q", None),
    "Medical Practices": ("Q", None),
    "Veterinary Services": ("Q", None),
    "Alternative Medicine": ("Q", None),
    # R — Arts & recreation
    "Spectator Sports": ("R", None),
    "Entertainment Providers": ("R", None),
    "Museums, Historical Sites, and Zoos": ("R", None),
    "Recreational Facilities": ("R", None),
    "Musicians": ("R", None),
    "Performing Arts": ("R", None),
    "Artists and Writers": ("R", None),
    "Gambling Facilities and Casinos": ("R", None),
    # S — Other services
    "Non-profit Organizations": ("S", None),
    "Civic and Social Organizations": ("S", None),
    "Consumer Services": ("S", None),
    "Philanthropic Fundraising Services": ("S", None),
    "Fundraising": ("S", None),
    "Religious Institutions": ("S", None),
}

SPLIT_SECTIONS = {"B", "C", "D", "E", "H", "J", "M"}
RESIDUAL = {
    "B": "B — Other mining",        "C": "C — Other manufacturing",
    "D": "D — Other energy",        "E": "E — Other environmental",
    "H": "H — Other transport",     "J": "J — Other information",
    "M": "M — Other professional",
}

# NACE-1 section headings, so every stored value is self-describing.
SECTION_LABEL = {
    "A": "A — Agriculture, forestry and fishing",
    "B": "B — Mining and quarrying",
    "C": "C — Manufacturing",
    "D": "D — Energy and utilities",
    "E": "E — Water and environmental services",
    "F": "F — Construction",
    "G": "G — Wholesale and retail trade",
    "H": "H — Transportation and storage",
    "I": "I — Accommodation and food service",
    "J": "J — Information and communication",
    "K": "K — Finance and insurance",
    "L": "L — Real estate",
    "M": "M — Professional, scientific and technical",
    "N": "N — Administrative and support services",
    "O": "O — Public administration",
    "P": "P — Education",
    "Q": "Q — Human health and social work",
    "R": "R — Arts, entertainment and recreation",
    "S": "S — Other services",
}

# Flatten to direct lookups (mirrors the original .map(dict) idiom). Values are
# uniform, self-describing strings: divisions carry their parent section letter
# ("C28 — Machinery Manufacturing"), bare sections carry the full NACE-1 heading
# ("K — Finance and insurance"), residuals keep section + description. The
# relabel is bijective, so the cross-sector equality test is unaffected.
# Unmapped industry strings are absent from these dicts -> .map yields NaN.
nace2_proxy_map = {}
nace2_proxy_nace1_map = {}
for _ind, (_sec, _div) in industry_to_sector.items():
    nace2_proxy_nace1_map[_ind] = SECTION_LABEL[_sec]
    if _div:
        nace2_proxy_map[_ind] = _sec + _div            # "C" + "28 — ..." -> "C28 — ..."
    elif _sec in SPLIT_SECTIONS:
        nace2_proxy_map[_ind] = RESIDUAL[_sec]
    else:
        nace2_proxy_map[_ind] = SECTION_LABEL[_sec]

firm_master_ipcei["nace2_proxy"] = firm_master_ipcei["industry"].map(nace2_proxy_map)
firm_master_ipcei["nace2_proxy_nace1"] = firm_master_ipcei["industry"].map(nace2_proxy_nace1_map)

# ── Diagnostics ───────────────────────────────────────────────────────────────
partnership_firms = set(p["source"]).union(set(p["target"]))

cov_partners = firm_master_ipcei[
    firm_master_ipcei["Linkedin_name"].isin(partnership_firms)
]["nace2_proxy"].notna().mean()

cov_treated = firm_master_ipcei[
    firm_master_ipcei["treated_any"].fillna(0).astype(int) == 1
]["nace2_proxy"].notna().mean()

print(f"Coverage among partnership firms: {cov_partners:.1%}")
print(f"Coverage among treated firms:     {cov_treated:.1%}")

print(f"\nnace2_proxy distribution (treated firms):")
print(
    firm_master_ipcei[firm_master_ipcei["treated_any"].fillna(0).astype(int)==1]
    ["nace2_proxy"].value_counts(dropna=False)
)

print(f"\nnace2_proxy_nace1 distribution (treated firms):")
print(
    firm_master_ipcei[firm_master_ipcei["treated_any"].fillna(0).astype(int)==1]
    ["nace2_proxy_nace1"].value_counts(dropna=False)
)

print("✓ nace2_proxy created")


COMPUTING NACE2_PROXY
Coverage among partnership firms: 94.6%
Coverage among treated firms:     100.0%

nace2_proxy distribution (treated firms):
nace2_proxy
D35 — Utilities                      26
C28 — Machinery Manufacturing        23
C26 — Electronics Manufacturing      22
C29 — Motor Vehicle Manufacturing    22
C20 — Chemical Manufacturing         17
D06 — Oil and Gas                    13
J62 — IT Services                     6
B07 — Mining                          5
C — Other manufacturing               4
M72 — Research Services               4
H49 — Land Transport                  3
E38 — Environmental Services          2
M — Other professional                1
K — Finance and insurance             1
G — Wholesale and retail trade        1
Name: count, dtype: int64

nace2_proxy_nace1 distribution (treated firms):
nace2_proxy_nace1
C — Manufacturing                             88
D — Energy and utilities                      39
J — Information and communication              6
B

In [71]:
# ============================================================
# Yearly collaboration counts (2020–2025)
# ============================================================

# Ensure datetime
partnership_master['post_date'] = pd.to_datetime(
    partnership_master['post_date'],
    errors='coerce'
)

# ----------------------------------------
# Stack collaborations (source + target)
# ----------------------------------------

src = partnership_master[['source', 'collaboration_id', 'post_date']] \
    .rename(columns={'source': 'Linkedin_name'})

tgt = partnership_master[['target', 'collaboration_id', 'post_date']] \
    .rename(columns={'target': 'Linkedin_name'})

collab_long = pd.concat([src, tgt], ignore_index=True)

# Extract year
collab_long['year'] = collab_long['post_date'].dt.year

# Restrict to relevant years
years = list(range(2020, 2026))
collab_long = collab_long[collab_long['year'].isin(years)]

# ----------------------------------------
# Count collaborations per firm-year
# ----------------------------------------

collab_counts = (
    collab_long
    .groupby(['Linkedin_name', 'year'])['collaboration_id']
    .nunique()
    .reset_index(name='collab_count')
)

# Pivot to wide
collab_wide = collab_counts.pivot(
    index='Linkedin_name',
    columns='year',
    values='collab_count'
)

collab_wide = collab_wide.fillna(0).astype(int)

# Rename columns
collab_wide.columns = [f'collab_count_{int(c)}' for c in collab_wide.columns]

collab_wide = collab_wide.reset_index()

print(f"✓ Created yearly counts for {len(collab_wide):,} firms")

# ----------------------------------------
# Merge into firm master
# ----------------------------------------

firm_master_ipcei = firm_master_ipcei.merge(
    collab_wide,
    on='Linkedin_name',
    how='left'
)

# Fill missing firms with zero
for y in years:
    col = f'collab_count_{y}'
    if col not in firm_master_ipcei:
        firm_master_ipcei[col] = 0
    else:
        firm_master_ipcei[col] = firm_master_ipcei[col].fillna(0)

# ----------------------------------------
# Log transforms
# ----------------------------------------

for y in years:
    firm_master_ipcei[f'log_collab_count_{y}'] = np.log(
        firm_master_ipcei[f'collab_count_{y}'] + 1
    )

print("✓ Log variables created")

# ============================================================
# Pre-treatment windows derived from yearly counts
# ============================================================

print("\nDeriving baseline windows...")

# Battery baseline: 2020
firm_master_ipcei['collab_pretreat_battery'] = firm_master_ipcei['collab_count_2020']
firm_master_ipcei['log_collab_pretreat_battery'] = np.log(
    firm_master_ipcei['collab_pretreat_battery'] + 1
)

# Hydrogen baseline: 2021
firm_master_ipcei['collab_pretreat_hydrogen'] = firm_master_ipcei['collab_count_2021']
firm_master_ipcei['log_collab_pretreat_hydrogen'] = np.log(
    firm_master_ipcei['collab_pretreat_hydrogen'] + 1
)

print("✓ Baseline variables created")

# ============================================================
# Diagnostics
# ============================================================

print("\n" + "="*80)
print("COLLABORATION DISTRIBUTION CHECK")
print("="*80)

for y in years:
    col = f'collab_count_{y}'
    print(f"\n{col}:")
    print(f"  Mean: {firm_master_ipcei[col].mean():.2f}")
    print(f"  Median: {firm_master_ipcei[col].median():.0f}")
    print(f"  Zero share: {(firm_master_ipcei[col] == 0).mean():.2%}")

print("\n✓ Collaboration variables ready")

print("\n" + "="*80)
print("TREATED FIRMS — BASELINE COLLABORATION")
print("="*80)

treated = firm_master_ipcei[firm_master_ipcei["treated_any"] == 1]

for y in [2020, 2021]:
    col = f'collab_count_{y}'
    print(f"\n{col}:")
    print(f"  Mean: {treated[col].mean():.2f}")
    print(f"  Median: {treated[col].median():.0f}")
    print(f"  Zero share: {(treated[col] == 0).mean():.2%}")

✓ Created yearly counts for 217,109 firms
✓ Log variables created

Deriving baseline windows...
✓ Baseline variables created

COLLABORATION DISTRIBUTION CHECK

collab_count_2020:
  Mean: 0.30
  Median: 0
  Zero share: 91.53%

collab_count_2021:
  Mean: 0.68
  Median: 0
  Zero share: 85.23%

collab_count_2022:
  Mean: 1.09
  Median: 0
  Zero share: 79.53%

collab_count_2023:
  Mean: 1.76
  Median: 0
  Zero share: 71.06%

collab_count_2024:
  Mean: 1.97
  Median: 0
  Zero share: 74.49%

collab_count_2025:
  Mean: 1.73
  Median: 0
  Zero share: 77.60%

✓ Collaboration variables ready

TREATED FIRMS — BASELINE COLLABORATION

collab_count_2020:
  Mean: 17.82
  Median: 2
  Zero share: 40.00%

collab_count_2021:
  Mean: 44.76
  Median: 13
  Zero share: 22.67%


In [72]:
# add extensive margin

firm_master_ipcei["any_collab_2020"] = (
    firm_master_ipcei["collab_count_2020"] > 0
).astype(int)

firm_master_ipcei["any_collab_2021"] = (
    firm_master_ipcei["collab_count_2021"] > 0
).astype(int)

In [73]:
# ============================================================
# CENTRALITIES — Degree + Betweenness (yearly, deduplicated)
# ============================================================

print("\n" + "="*80)
print("COMPUTING NETWORK CENTRALITIES")
print("="*80)

# Ensure datetime
partnership_master['post_date'] = pd.to_datetime(
    partnership_master['post_date'], errors='coerce'
)

# Extract year
partnership_master['year'] = partnership_master['post_date'].dt.year

years = list(range(2020, 2026))

# Deduplicate by unordered pair within year
pm = partnership_master.copy()
pm['pair'] = pm.apply(
    lambda x: tuple(sorted([x['source'], x['target']])),
    axis=1
)

pm = pm.drop_duplicates(subset=['pair', 'year'])

# Prepare storage
centrality_results = []

for year in years:

    print(f"Processing network for {year}...")

    df_year = pm[pm['year'] == year]

    # Build graph
    G = nx.from_pandas_edgelist(df_year, source='source', target='target')

    # Degree centrality
    degree = nx.degree_centrality(G)

    # Betweenness centrality (approximation for speed)
    betweenness = nx.betweenness_centrality(G, k=min(500, len(G.nodes())), seed=42)

    df_cent = pd.DataFrame({
        'Linkedin_name': list(G.nodes()),
        f'degree_centrality_{year}': [degree.get(n, 0) for n in G.nodes()],
        f'betweenness_centrality_{year}': [betweenness.get(n, 0) for n in G.nodes()]
    })

    centrality_results.append(df_cent)

# Merge all years
from functools import reduce

centralities = reduce(
    lambda left, right: pd.merge(left, right, on='Linkedin_name', how='outer'),
    centrality_results
)

# Merge into firm master
firm_master_ipcei = firm_master_ipcei.merge(
    centralities,
    on='Linkedin_name',
    how='left'
)

# Fill missing with zero
for col in centralities.columns:
    if col != "Linkedin_name":
        firm_master_ipcei[col] = firm_master_ipcei[col].fillna(0)

print("✓ Centralities merged into firm master")


COMPUTING NETWORK CENTRALITIES
Processing network for 2020...
Processing network for 2021...
Processing network for 2022...
Processing network for 2023...
Processing network for 2024...
Processing network for 2025...
✓ Centralities merged into firm master


In [74]:
# ============================================================
# LOG CENTRALITIES
# ============================================================

for y in years:
    deg = f"degree_centrality_{y}"
    bet = f"betweenness_centrality_{y}"

    # safe even when values are 0
    firm_master_ipcei[f"log_{deg}"] = np.log1p(firm_master_ipcei[deg])
    firm_master_ipcei[f"log_{bet}"] = np.log1p(firm_master_ipcei[bet])

print("✓ Log centralities created")

✓ Log centralities created


In [75]:
# ============================================================
# TECH COLLABORATION COUNTS
# ============================================================

print("\n" + "="*80)
print("COMPUTING TECH COLLAB COUNTS")
print("="*80)

# If running standalone, re-run the p = partnership_master.copy() block above

techs = ["Hydrogen", "Battery"]
tech_results = []

for tech in techs:
    df_tech = p[p[tech] == 1].copy()

    src = df_tech[["source","collaboration_id","year"]].rename(columns={"source":"Linkedin_name"})
    tgt = df_tech[["target","collaboration_id","year"]].rename(columns={"target":"Linkedin_name"})
    stacked = pd.concat([src, tgt], ignore_index=True)
    stacked = stacked[stacked["year"].isin(years)]

    counts = (
        stacked.groupby(["Linkedin_name","year"])["collaboration_id"]
               .nunique().reset_index(name=f"{tech.lower()}_collabs")
    )
    wide = counts.pivot(
        index="Linkedin_name", columns="year",
        values=f"{tech.lower()}_collabs"
    ).fillna(0)
    wide.columns = [f"{tech.lower()}_collabs_{int(c)}" for c in wide.columns]
    tech_results.append(wide.reset_index())

tech_df = reduce(
    lambda left, right: pd.merge(left, right, on="Linkedin_name", how="outer"),
    tech_results
)

firm_master_ipcei = firm_master_ipcei.merge(tech_df, on="Linkedin_name", how="left")
# Fill everything EXCEPT the sector-proxy columns: genuinely unclassified firms
# must stay NaN there (never become "0"), so no downstream code can mistake
# "unknown sector" for a real one. The cross-sector flag's _is_known guard
# treats NaN as unclassified.
_protect = ["nace2_proxy", "nace2_proxy_nace1"]
_fill_cols = [col for col in firm_master_ipcei.columns if col not in _protect]
firm_master_ipcei[_fill_cols] = firm_master_ipcei[_fill_cols].fillna(0)

firm_master_ipcei["hydrogen_collabs_pre"] = firm_master_ipcei.get("hydrogen_collabs_2020", 0)
firm_master_ipcei["battery_collabs_pre"]  = firm_master_ipcei.get("battery_collabs_2020",  0)

print("✓ Tech collaboration variables created")


COMPUTING TECH COLLAB COUNTS
✓ Tech collaboration variables created


/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_60406/2448529930.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  firm_master_ipcei["hydrogen_collabs_pre"] = firm_master_ipcei.get("hydrogen_collabs_2020", 0)
/var/folders/1w/t9ryrq_57m77f4wzbw78745m044mx9/T/ipykernel_60406/2448529930.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  firm_master_ipcei["battery_collabs_pre"]  = firm_master_ipcei.get("battery_collabs_2020",  0)


#### 3. Prepare panels and save

In [76]:
# Check for duplicate _x/_y columns
dup_cols = [c for c in firm_master_ipcei.columns
            if c.endswith("_x") or c.endswith("_y")]
print(dup_cols)

# The _y versions are from the most recent merge (geographic/tech chunk rerun)
# Keep _y, drop _x
cols_to_drop = [c for c in firm_master_ipcei.columns if c.endswith("_x")]
cols_to_rename = {c: c.replace("_y", "") for c in firm_master_ipcei.columns
                  if c.endswith("_y")}

firm_master_ipcei = firm_master_ipcei.drop(columns=cols_to_drop)
firm_master_ipcei = firm_master_ipcei.rename(columns=cols_to_rename)

print(f"Dropped: {cols_to_drop}")
print(f"Renamed: {cols_to_rename}")

# Verify no more duplicates
print([c for c in firm_master_ipcei.columns if c.endswith("_x") or c.endswith("_y")])

[]
Dropped: []
Renamed: {}
[]


In [77]:
# ============================================================
# Per-technology first IPCEI approval dates
# ============================================================

_evt = firm_master_ipcei_events.copy()
_evt["ipcei_post_date"] = pd.to_datetime(_evt["ipcei_post_date"], errors="coerce")
_evt["_type"] = _evt["ipcei_type"].astype(str).str.strip().str.lower()

_first_by_tech = (
    _evt.dropna(subset=["ipcei_post_date"])
        .groupby(["Linkedin_name", "_type"])["ipcei_post_date"]
        .min()
        .unstack("_type")
)

for _tech in ["battery", "hydrogen"]:
    _col = f"first_{_tech}_date"
    firm_master_ipcei[_col] = (
        firm_master_ipcei["Linkedin_name"].map(_first_by_tech[_tech])
        if _tech in _first_by_tech.columns else pd.NaT
    )
    firm_master_ipcei[_col] = pd.to_datetime(firm_master_ipcei[_col], errors="coerce")

print("Derived per-technology approval dates:")
print(f"  first_battery_date non-null:  {firm_master_ipcei['first_battery_date'].notna().sum()}")
print(f"  first_hydrogen_date non-null: {firm_master_ipcei['first_hydrogen_date'].notna().sum()}")


Derived per-technology approval dates:
  first_battery_date non-null:  47
  first_hydrogen_date non-null: 108


In [78]:
# ============================================================
# SAVE extended firm master
# ============================================================

print("\nSaving datasets")

# Save firm master
save_df(
    firm_master_ipcei,
    "interim",
    "combined",
    "firm_master_ipcei_corrected_extended_jan26.csv"
)

print("✓ Extended firm master saved")


Saving datasets
✓ Extended firm master saved


In [79]:
# =========================
# Extend firm_month_outcomes
# =========================

print("\n" + "="*80)
print("Extending firm months coutomces")
print("="*80)

# ── Prep ──────────────────────────────────────────────────────────────────────
firm_month_outcomes["month"] = pd.to_datetime(firm_month_outcomes["month"])

# Drop any previously computed collab columns to avoid duplicates on re-run
cols_to_drop = [c for c in firm_month_outcomes.columns
                if c.startswith("n_collabs_") and c != "n_collabs"]
if cols_to_drop:
    firm_month_outcomes = firm_month_outcomes.drop(columns=cols_to_drop)
    print(f"Dropped existing columns: {cols_to_drop}")

# Ensure month column on p is present (may already exist from Geographic chunk)
if "month" not in p.columns:
    p["month"] = p["post_date"].dt.to_period("M").dt.to_timestamp()

# ── Stack endpoints helper ────────────────────────────────────────────────────
def _stack_and_count(df_sub, count_name):
    src = (df_sub[["month","collaboration_id","source"]]
           .rename(columns={"source":"Linkedin_name"}))
    tgt = (df_sub[["month","collaboration_id","target"]]
           .rename(columns={"target":"Linkedin_name"}))
    ft  = pd.concat([src, tgt], ignore_index=True)
    ft  = ft.dropna(subset=["Linkedin_name","month","collaboration_id"])
    return (
        ft.groupby(["Linkedin_name","month"])["collaboration_id"]
          .nunique()
          .reset_index(name=count_name)
    )

# ── 1. Tech-specific collabs ──────────────────────────────────────────────────
for tech in ["Hydrogen", "Battery"]:
    col_name = f"n_collabs_{tech.lower()}"
    fmo_tech = _stack_and_count(p[p[tech] == 1].copy(), col_name)
    firm_month_outcomes = firm_month_outcomes.merge(
        fmo_tech, on=["Linkedin_name","month"], how="left"
    )
    firm_month_outcomes[col_name] = firm_month_outcomes[col_name].fillna(0).astype(int)
    print(f"✓ {col_name}: {(firm_month_outcomes[col_name] > 0).sum():,} obs with >0")

# ── 2. EU ecosystem (any EU operation, both firms) ────────────────────────────
fmo_eu = _stack_and_count(p[p["is_eu_ecosystem"]].copy(), "n_collabs_eu")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu"] = (
    firm_month_outcomes["n_collabs_eu"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu: {(firm_month_outcomes['n_collabs_eu'] > 0).sum():,} obs with >0")

# ── 3. EU primary HQ (both firms' primary country is EU) ─────────────────────
fmo_eu_primary = _stack_and_count(p[p["is_eu_primary"]].copy(), "n_collabs_eu_primary")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu_primary, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu_primary"] = (
    firm_month_outcomes["n_collabs_eu_primary"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu_primary: {(firm_month_outcomes['n_collabs_eu_primary'] > 0).sum():,} obs with >0")

# ── 4. EU cross-border (both primary EU, different member states) ─────────────
fmo_eu_cb = _stack_and_count(p[p["is_eu_crossborder"]].copy(), "n_collabs_eu_crossborder")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_eu_cb, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_eu_crossborder"] = (
    firm_month_outcomes["n_collabs_eu_crossborder"].fillna(0).astype(int)
)
print(f"✓ n_collabs_eu_crossborder: {(firm_month_outcomes['n_collabs_eu_crossborder'] > 0).sum():,} obs with >0")

# ── 5. Non-EU (one primary non EU) ────────────────────────────
fmo_non_eu = _stack_and_count(p[p["is_non_eu"]].copy(), "n_collabs_non_eu")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_non_eu, on=["Linkedin_name","month"], how="left"
)
firm_month_outcomes["n_collabs_non_eu"] = (
    firm_month_outcomes["n_collabs_non_eu"].fillna(0).astype(int)
)
print(f"✓ n_collabs_non_eu: {(firm_month_outcomes['n_collabs_non_eu'] > 0).sum():,} obs with >0")

# ── 6. Domestic (same primary country) ────────────────────────────
fmo_domestic   = _stack_and_count(p[p["is_domestic"]].copy(), "n_collabs_domestic")
firm_month_outcomes = firm_month_outcomes.merge(
    fmo_domestic, on=["Linkedin_name", "month"], how="left"
)
firm_month_outcomes["n_collabs_domestic"] = (
    firm_month_outcomes["n_collabs_domestic"].fillna(0).astype(int)
)
print(f"✓ n_collabs_domestic: {(firm_month_outcomes['n_collabs_domestic'] > 0).sum():,} obs with >0")

# ── 7. Cross-sector (different nace2_proxy, both endpoints classified) ────────
# nace2_proxy: NACE-2 division for IPCEI-relevant sections, per-section residual
# for split-section orphans, NACE-1 letter elsewhere. The blanket fillna(0)
# above turns unclassified firms into "0", so a firm counts only when its
# sector is genuinely known (_is_known). nace2_proxy_nace1 is the pure
# NACE-1-letter collapse used for the robustness outcome below.
nace2_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["nace2_proxy"]
))
nace1_map = dict(zip(
    firm_master_ipcei["Linkedin_name"],
    firm_master_ipcei["nace2_proxy_nace1"]
))

def _is_known(s):
    if pd.isna(s):
        return False
    return str(s).strip() not in ("0", "nan", "none", "")

p["src_nace2"] = p["source"].map(nace2_map)
p["tgt_nace2"] = p["target"].map(nace2_map)
p["src_known"] = p["src_nace2"].map(_is_known)
p["tgt_known"] = p["tgt_nace2"].map(_is_known)
p["is_crosssector"] = (
    p["src_known"] &
    p["tgt_known"] &
    (p["src_nace2"] != p["tgt_nace2"])
)
p["is_withinsector"] = (
    p["src_known"] &
    p["tgt_known"] &
    (p["src_nace2"] == p["tgt_nace2"])
)

# NACE-1-resolution cross-sector (robustness): same rule on nace2_proxy_nace1
p["src_nace1"] = p["source"].map(nace1_map)
p["tgt_nace1"] = p["target"].map(nace1_map)
p["src_nace1_known"] = p["src_nace1"].map(_is_known)
p["tgt_nace1_known"] = p["tgt_nace1"].map(_is_known)
p["is_crosssector_nace1"] = (
    p["src_nace1_known"] &
    p["tgt_nace1_known"] &
    (p["src_nace1"] != p["tgt_nace1"])
)

fmo_xs = _stack_and_count(p[p["is_crosssector"]].copy(),  "n_collabs_crosssector")
fmo_ws = _stack_and_count(p[p["is_withinsector"]].copy(), "n_collabs_withinsector")
fmo_xs_n1 = _stack_and_count(p[p["is_crosssector_nace1"]].copy(), "n_collabs_crosssector_nace1")

firm_month_outcomes = (
    firm_month_outcomes
    .merge(fmo_xs, on=["Linkedin_name","month"], how="left")
    .merge(fmo_ws, on=["Linkedin_name","month"], how="left")
    .merge(fmo_xs_n1, on=["Linkedin_name","month"], how="left")
)
firm_month_outcomes["n_collabs_crosssector"]  = (
    firm_month_outcomes["n_collabs_crosssector"].fillna(0).astype(int)
)
firm_month_outcomes["n_collabs_withinsector"] = (
    firm_month_outcomes["n_collabs_withinsector"].fillna(0).astype(int)
)
firm_month_outcomes["n_collabs_crosssector_nace1"] = (
    firm_month_outcomes["n_collabs_crosssector_nace1"].fillna(0).astype(int)
)
print(f"✓ n_collabs_crosssector:  {(firm_month_outcomes['n_collabs_crosssector'] > 0).sum():,} obs with >0")
print(f"✓ n_collabs_withinsector: {(firm_month_outcomes['n_collabs_withinsector'] > 0).sum():,} obs with >0")
print(f"✓ n_collabs_crosssector_nace1: {(firm_month_outcomes['n_collabs_crosssector_nace1'] > 0).sum():,} obs with >0")

# ── Collaboration OUTSIDE the funded technology ───────────────────────────────
# Total minus technology-tagged. Used as the redirection benchmark: if approval
# redirects rather than expands, this stays flat while n_collabs_{tech} rises.
# Only ~2% of collaborations carry both tech tags, so the split is near-partition.
for _t in ("battery", "hydrogen"):
    firm_month_outcomes[f"n_collabs_nonfunded_{_t}"] = (
        firm_month_outcomes["n_collabs"] - firm_month_outcomes[f"n_collabs_{_t}"]
    ).clip(lower=0).astype(int)
    print(f"✓ n_collabs_nonfunded_{_t}: "
          f"{(firm_month_outcomes[f'n_collabs_nonfunded_{_t}'] > 0).sum():,} obs with >0")

# ── 8. Tech-tagged composition outcomes ───────────────────────────────────
comp_flags = {
    "eu_crossborder": "is_eu_crossborder",
    "non_eu":         "is_non_eu",
    "domestic":       "is_domestic",
    "crosssector":    "is_crosssector",
    "withinsector":   "is_withinsector",
    "crosssector_nace1": "is_crosssector_nace1",
}
for tech in ["Hydrogen", "Battery"]:
    tcol  = tech.lower()
    tmask = (p[tech] == 1)
    for short, flag in comp_flags.items():
        col = f"n_collabs_{short}_{tcol}"
        fmo = _stack_and_count(p[tmask & p[flag]].copy(), col)
        firm_month_outcomes = firm_month_outcomes.merge(
            fmo, on=["Linkedin_name", "month"], how="left"
        )
        firm_month_outcomes[col] = firm_month_outcomes[col].fillna(0).astype(int)
        print(f"OK {col}: {(firm_month_outcomes[col] > 0).sum():,} obs with >0")
    col = f"n_collabs_crosssector_eu_{tcol}"
    fmo = _stack_and_count(p[tmask & p["is_crosssector"] & p["is_eu_primary"]].copy(), col)
    firm_month_outcomes = firm_month_outcomes.merge(
        fmo, on=["Linkedin_name", "month"], how="left"
    )
    firm_month_outcomes[col] = firm_month_outcomes[col].fillna(0).astype(int)
    print(f"OK {col}: {(firm_month_outcomes[col] > 0).sum():,} obs with >0")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nfirm_month_outcomes final shape: {firm_month_outcomes.shape}")
print(f"Columns: {list(firm_month_outcomes.columns)}")

# ── Save ──────────────────────────────────────────────────────────────────────
save_df(firm_month_outcomes, "interim", "combined",
        "firm_month_outcomes_extended_jan26.csv")
print("✓ firm_month_outcomes_extended saved")


Extending firm months coutomces
✓ n_collabs_hydrogen: 182,081 obs with >0
✓ n_collabs_battery: 188,679 obs with >0
✓ n_collabs_eu: 369,542 obs with >0
✓ n_collabs_eu_primary: 338,137 obs with >0
✓ n_collabs_eu_crossborder: 166,465 obs with >0
✓ n_collabs_non_eu: 452,375 obs with >0
✓ n_collabs_domestic: 283,535 obs with >0
✓ n_collabs_crosssector:  734,001 obs with >0
✓ n_collabs_withinsector: 352,660 obs with >0
✓ n_collabs_crosssector_nace1: 699,331 obs with >0
OK n_collabs_eu_crossborder_hydrogen: 40,653 obs with >0
OK n_collabs_non_eu_hydrogen: 95,430 obs with >0
OK n_collabs_domestic_hydrogen: 70,175 obs with >0
OK n_collabs_crosssector_hydrogen: 145,909 obs with >0
OK n_collabs_withinsector_hydrogen: 71,917 obs with >0
OK n_collabs_crosssector_nace1_hydrogen: 139,604 obs with >0
OK n_collabs_crosssector_eu_hydrogen: 75,996 obs with >0
OK n_collabs_eu_crossborder_battery: 29,915 obs with >0
OK n_collabs_non_eu_battery: 82,413 obs with >0
OK n_collabs_domestic_battery: 47,233 obs 

In [80]:
# How many collaborations have more than two firms:

firms_per_collab = (
    pd.concat([
        p[["collaboration_id", "source"]].rename(columns={"source": "f"}),
        p[["collaboration_id", "target"]].rename(columns={"target": "f"}),
    ])
    .groupby("collaboration_id")["f"].nunique()
)
print(firms_per_collab.describe())
print("share with >2 firms:", round((firms_per_collab > 2).mean(), 3))

count    701566.000000
mean          2.924334
std           2.214854
min           1.000000
25%           2.000000
50%           2.000000
75%           3.000000
max          57.000000
Name: f, dtype: float64
share with >2 firms: 0.374
